In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from pathlib import Path

# --------------------------
# 1. Config
# --------------------------
base_dir = Path("/Users/ewellmeyer/Documents/research/weights")

# Select which channel counts to include
channels = [8, 16, 32, 64, 128, 256]

# gn_groups used per channel during training (all 1 by default for unet6R runs)
gn_per_ch = {ch: 1 for ch in channels}

# Fixed training parameters (must match what train_pr_dpdk.py used)
k_size    = 3
num_bins  = 64
dP_min    = -700
dP_max    = 1200

def ens_dir(ch, suffix=""):
    gn = gn_per_ch[ch]
    name = (
        f"unet_ens_HG789_PR_dPdK_Softmax_unet6R_ch{ch}_k{k_size}_"
        f"128x_dPbins{num_bins}_gn{gn}_dpmin{dP_min}_dPmax{dP_max}_sigma0.6"
        f"{suffix}"
    )
    return base_dir / name

# --------------------------
# 2. Load results
# --------------------------
def load_pct(suffix=""):
    out = []
    for ch in channels:
        npz_path = ens_dir(ch, suffix) / "softmax_ensemble_analysis_arrays.npz"
        try:
            d = np.load(npz_path)
            idx = d['test_indices']
            rmse_model    = d['rmse_softmax_mean'][idx]
            rmse_baseline = d['rmse_ppe'][idx]
            pct = (rmse_baseline - rmse_model) / rmse_baseline * 100
            out.append(np.median(pct))
        except FileNotFoundError:
            out.append(np.nan)
            print(f"Missing results for ch={ch} (suffix='{suffix}'): {npz_path}")
    return out

pct_improvement     = load_pct("")
pct_improvement_dr  = load_pct("_dr0.05")
pct_improvement_dr1 = load_pct("_dr0.1")

# --------------------------
# 3. Plot
# --------------------------
sns.set_context("paper", font_scale=1.4)
sns.set_style("ticks")

fig, ax = plt.subplots(figsize=(8, 6))

c_soft = '#1f77b4'
c_dr   = '#d62728'
c_dr1  = '#2ca02c'

ax.axhline(0, color='gray', linestyle=':', linewidth=1.5, alpha=0.8, zorder=1)

ax.plot(channels, pct_improvement,     color=c_soft, linestyle='-', marker='o', lw=2, ms=6)
ax.plot(channels, pct_improvement_dr,  color=c_dr,   linestyle='-', marker='s', lw=2, ms=6)
ax.plot(channels, pct_improvement_dr1, color=c_dr1,  linestyle='-', marker='^', lw=2, ms=6)

ax.set_xlabel("Network Channels per Layer", fontsize=12)
ax.set_ylabel("Median % Improvement over PPE Train Mean", fontsize=12, fontweight='bold')
ax.set_xscale('log', base=2)
ax.set_xticks(channels)
ax.set_xticklabels(channels)
ax.set_xlim(7, 275)
ax.set_ylim(10, 30)
ax.grid(True, which='major', linestyle='--', alpha=0.4)
sns.despine(ax=ax)

legend_elements = [
    Line2D([0], [0], color=c_soft, lw=3, marker='o', ms=6, label='Base Model (HadGEM Test)'),
    Line2D([0], [0], color=c_dr,   lw=3, marker='s', ms=6, label='Dropout 0.05 (HadGEM Test)'),
    Line2D([0], [0], color=c_dr1,  lw=3, marker='^', ms=6, label='Dropout 0.1 (HadGEM Test)'),
    Line2D([0], [0], color='gray', linestyle=':', lw=2, label='PPE Train Mean Baseline'),
]
ax.legend(handles=legend_elements, frameon=False, fontsize=11, loc='upper left')

plt.tight_layout()
plt.show()